# Legacy results: diagnostic analysis

These figures summarize recorded timings, not validated cross-system performance.
Null timings are missing observations, not zero-duration successes. Successful-only
means can compare different query mixes and must be interpreted alongside missing rates.

Exact throughput is unavailable: the logs have no start/finish timestamps, actual
measurement boundaries, or durations for missing observations. Configured duration
is not a substitute for an observed measurement interval. No throughput is plotted.

`legacy-runs.json` preserves the original paper selection; scale factors come from
the old notebook because the run metadata omitted them. Worker counts are validated.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from jsonbench.analysis import summarize_run

root = Path.cwd()
if not (root / "legacy-runs.json").exists():
    raise RuntimeError("Run this notebook from the repository root")
manifest = json.loads((root / "legacy-runs.json").read_text())
runs, query_rows = [], []
for entry in manifest:
    summary = summarize_run(
        root / "jsonbench/results" / entry["backend"] / entry["run_id"],
        expected_workers=entry["workers"],
    )
    queries = summary.pop("queries")
    runs.append({**entry, **summary})
    for name, metrics in queries.items():
        query_rows.append({**entry, "query": name, **metrics})
run_table = pd.DataFrame(runs)
query_table = pd.DataFrame(query_rows)
run_table["missing_percent"] = 100 * run_table["missing_timings"] / run_table["attempts"]
run_table


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), layout="constrained")
for column, scale in enumerate([10, 100, 200]):
    for backend in ["mongodb", "couchbase"]:
        rows = run_table[(run_table.backend == backend) & (run_table.scale_factor == scale)].sort_values("workers")
        axes[0, column].plot(rows.workers, rows.mean_seconds, marker="o", label=backend)
        axes[1, column].plot(rows.workers, rows.missing_percent, marker="o", label=backend)
    axes[0, column].set_title(f"Scale factor {scale}")
    axes[0, column].set_ylabel("Mean numeric timing (seconds)")
    axes[1, column].set_ylabel("Missing timings (%)")
    axes[1, column].set_ylim(0, 105)
    for ax in axes[:, column]:
        ax.set_xlabel("Clients")
        ax.grid(alpha=0.2)
axes[0, 0].legend()
fig.suptitle("Diagnostic only: numeric means exclude missing observations")
plt.show()


## Per-query observations

Choose one scale and concurrency. Each backend is aggregated independently before
comparison; raw observations are never merged across systems. Absent bars indicate
no numeric observations, not zero latency. Consult the counts below.


In [ ]:
scale_factor, workers = 100, 64
selected = query_table[(query_table.scale_factor == scale_factor) & (query_table.workers == workers)]
selected.pivot(index="query", columns="backend", values="mean_seconds").plot.bar(
    ylabel="Mean numeric timing (seconds)",
    title=f"SF={scale_factor}, clients={workers}: numeric observations only",
    rot=0,
)
plt.show()
selected[["backend", "query", "attempts", "numeric_timings", "missing_timings", "mean_seconds"]]
